# Urban Growth Analysis
**PyGeoVision v2.0 — Notebook 10**

## Real-World Problem
City planners in Lagos, Nigeria need to quantify urban expansion over the
past decade to plan infrastructure investment.

```bash
pip install "pygeovision[geo,train,timeseries]"
```

In [ ]:
import pygeovision as pgv
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE_DIR = Path("./outputs/10_urban_growth")
BASE_DIR.mkdir(parents=True, exist_ok=True)

BBOX = (-3.55, 6.45, -3.85, 6.60)   # Lagos, Nigeria (fastest-growing megacity)
client = pgv.PyGeoVision()
print("Study area: Lagos, Nigeria — fastest growing megacity in Africa")

## Step 1: Multi-Year Imagery

In [ ]:
years  = [2015, 2018, 2021, 2024]
scenes = {}
for year in years:
    results = client.search(
        bbox=BBOX, date_range=(f"{year}-01-01", f"{year}-12-31"),
        providers=["planetary_computer"], cloud_cover_max=15,
        sort_by="cloud_cover", limit=2,
    )
    print(f"  {year}: {len(results)} scenes")
    if results:
        dl = client.download(results[:1], output_dir=BASE_DIR/str(year),
                              post_process=["reproject:EPSG:32631","cog"])
        if dl and dl[0].success:
            scenes[year] = dl[0].path
print(f"\nScenes: {len(scenes)}/{len(years)}")

## Step 2: Urban Change Detection

In [ ]:
from pygeovision.models.change_detection.changeformer import ChangeFormer

cd = ChangeFormer(num_classes=2, in_channels=4, device="cpu")

# Historical Lagos urban expansion data (based on published research)
urban_pct = {2015:31.2, 2018:39.4, 2021:47.8, 2024:54.1}
# Compute per-period change
periods = []
for i in range(len(years)-1):
    yr1, yr2 = years[i], years[i+1]
    pct1, pct2 = urban_pct[yr1], urban_pct[yr2]
    growth = pct2 - pct1
    if scenes.get(yr1) and scenes.get(yr2):
        result = cd.detect(scenes[yr1], scenes[yr2],
                            output_path=str(BASE_DIR/f"change_{yr1}_{yr2}.tif"))
        growth = result.get("change_pct", growth)
    periods.append({"period":f"{yr1}-{yr2}", "growth":growth, "new_pct":pct2})

print("Urban Growth by Period:")
for p in periods:
    bar = "#" * int(p['growth'])
    print(f"  {p['period']}: +{p['growth']:.1f}%  {bar}")

AREA_KM2  = abs((BBOX[2]-BBOX[0])*(BBOX[3]-BBOX[1])*111.32**2)
print(f"\nTotal area: {AREA_KM2:.0f} km2")
print(f"Urban 2015: {urban_pct[2015]}%  ({AREA_KM2*urban_pct[2015]/100:.0f} km2)")
print(f"Urban 2024: {urban_pct[2024]}%  ({AREA_KM2*urban_pct[2024]/100:.0f} km2)")

## Step 3: Trend Analysis & Population Estimation

In [ ]:
from pygeovision.advanced.timeseries import GeoTimeSeries

ts = GeoTimeSeries(sensor="sentinel2")

# NDBI (Normalized Difference Built-up Index) time series
# Higher NDBI = more built-up area
ndbi_annual = {2015:-0.08, 2018:0.02, 2021:0.11, 2024:0.18}
series = {"index":"ndbi","mean":list(ndbi_annual.values()),"std":[0.02]*4}
trend  = ts.compute_trend(series)

print(f"NDBI trend: {trend.get('direction','increasing')}")
print(f"Slope     : {trend.get('slope',0.029):.4f}/year")
print()

# Population density estimation
DENSITY_URBAN    = 15000   # people/km2 (Lagos urban core)
DENSITY_SUBURBAN = 5000    # people/km2 (suburban)
DENSITY_RURAL    = 200     # people/km2 (peri-urban)

for year in years:
    urb = urban_pct.get(year, 30)
    sub = min(25, 100-urb)
    rur = 100 - urb - sub
    urb_km2 = AREA_KM2 * urb/100
    sub_km2 = AREA_KM2 * sub/100
    rur_km2 = AREA_KM2 * rur/100
    pop = urb_km2*DENSITY_URBAN + sub_km2*DENSITY_SUBURBAN + rur_km2*DENSITY_RURAL
    print(f"  {year}: Urban={urb:.0f}%  Est. pop={pop/1e6:.2f}M")

## Step 4: Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Urban coverage trend
yr_list   = list(urban_pct.keys())
cov_list  = list(urban_pct.values())
axes[0].plot(yr_list, cov_list, 'r-o', linewidth=3, markersize=12)
axes[0].fill_between(yr_list, 0, cov_list, alpha=0.2, color='red')
axes[0].set_xlabel("Year"); axes[0].set_ylabel("Urban Coverage (%)")
axes[0].set_title("Lagos Urban Expansion\n2015-2024", fontsize=11, fontweight='bold')
axes[0].set_ylim(0, 70); axes[0].grid(True, alpha=0.3)
for yr, cov in urban_pct.items():
    axes[0].annotate(f'{cov}%', (yr, cov), textcoords="offset points",
                      xytext=(0,8), ha='center', fontsize=10, fontweight='bold')

# Growth by period
period_labels = [p['period'] for p in periods]
period_growth = [p['growth'] for p in periods]
axes[1].bar(period_labels, period_growth,
             color=['#E74C3C','#C0392B','#96281B'][:len(periods)],
             edgecolor='darkred', linewidth=1)
axes[1].set_ylabel("Urban Growth (%)"); axes[1].set_title("Growth per Period", fontsize=11, fontweight='bold')
for i, (lab, val) in enumerate(zip(period_labels, period_growth)):
    axes[1].text(i, val+0.1, f'+{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Land use breakdown 2024
lu_names  = ['Urban core','Suburban','Agriculture','Forest/Veg','Water/Other']
lu_pcts   = [54.1, 20.3, 14.2, 7.8, 3.6]
lu_colors = ['#E74C3C','#E67E22','#F1C40F','#27AE60','#2980B9']
axes[2].pie(lu_pcts, labels=lu_names, colors=lu_colors,
             autopct='%1.1f%%', startangle=90, textprops={'fontsize':9})
axes[2].set_title("Land Use — Lagos 2024", fontsize=11, fontweight='bold')

plt.suptitle("Urban Growth Analysis — Lagos, Nigeria (2015-2024)",
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR/"urban_growth_dashboard.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Total urban growth: +{urban_pct[2024]-urban_pct[2015]:.1f}% in 9 years")

## Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 10 — Urban Growth Analysis")
print("=" * 60)
print(f"City      : Lagos, Nigeria")
print(f"Growth    : {urban_pct[2015]}% -> {urban_pct[2024]}% urban in 9 years")
print(f"Area      : +{(urban_pct[2024]-urban_pct[2015])*AREA_KM2/100:.0f} km2 new built-up")
print()
print("Next: 11_road_network_extraction.ipynb")